# 02. 포트폴리오: 포지션 → QuantLib 상품 → NPV

| 단계 | 코드 |
|---|---|
| 포지션 | `positions/ust_book.yaml` (한 줄 = 한 거래, 커브는 이름으로) |
| 조립·가격 | `src/quanttool/portfolio.py` `Portfolio.build`, `npvs` |

> 데이터: `tests/fixtures/market_ust_sample.txt`는 **합성 샘플**(`날짜 키 값`)이다. 흐름을 익히는 용도이고 숫자로 시장을 해석하지 않는다.

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd
import QuantLib as ql

from quanttool.config import load_config
from quanttool.market import build_market
from quanttool.quotes import read_quotes

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
cfg = load_config(ROOT / "configs")
quotes = read_quotes(ROOT / "tests" / "fixtures" / "market_ust_sample.txt")  # 합성 샘플
ASOF = date(2026, 10, 23)

from quanttool.portfolio import Portfolio

market = build_market(ASOF, quotes[ASOF], cfg)

## 1. 포지션은 데이터
거래는 커브를 이름(`curve: UST`)으로만 가리킨다. 음수 notional = 매도.

In [ ]:
book = Portfolio.from_file(ROOT / "positions" / "ust_book.yaml")
pd.DataFrame([t.model_dump() for t in book.trades.values()]).set_index("id")

## 2. 시장에 붙여 가격
`build`가 거래마다 `ql.FixedRateBond`을 만들고 `DiscountingBondEngine(market['UST'])`를 붙인다. 같은 커브는 엔진 하나.

In [ ]:
book.build(market, cfg)
print("엔진:", list(book.engines))
display(book.npvs().round(2))
print("합계", round(book.npv(), 2))

## 3. QuantLib 직접 계산과 같은가

In [ ]:
with market.active():
    sch = ql.Schedule(
        ql.Date(15, 8, 2026),
        ql.Date(15, 8, 2036),
        ql.Period(ql.Semiannual),
        ql.UnitedStates(ql.UnitedStates.GovernmentBond),
        ql.Unadjusted,
        ql.Unadjusted,
        ql.DateGeneration.Backward,
        False,
    )
    direct = ql.FixedRateBond(
        1,
        100.0,
        sch,
        [0.04125],
        ql.ActualActual(ql.ActualActual.Bond),
        ql.Unadjusted,
        100.0,
        ql.Date(15, 8, 2026),
    )
    direct.setPricingEngine(
        ql.DiscountingBondEngine(ql.YieldTermStructureHandle(market.built["UST"].curve))
    )
    print(direct.NPV() * 1e7 / 100, book.npvs()["UST_10Y"])

## 4. 같은 포지션, 다른 날짜의 시장
포지션은 그대로 두고 다른 시장으로 다시 `build`한다.

In [ ]:
prev = build_market(date(2026, 10, 22), quotes[date(2026, 10, 22)], cfg)
now = book.npvs()
before = book.build(prev, cfg).npvs()
book.build(market, cfg)
pd.DataFrame({"10/22": before, "10/23": now, "손익": now - before}).round(2)